# M3: fundamentals lab

Eleven short experiments on the first 1,000 training people (81,645 photos). For each one:

1. I write what I expect under its heading in `docs/lab/notes.md`, before running anything.
2. I run it here and save one plot with `save(fig, "L1")`, which writes `docs/lab/L1.png`.
3. I add what happened, and why, to the notes.

I score on val; test stays unopened until M6. The notes and plots go in the public repo, so I never
plot or save a face.

An epoch takes about 90 s here (about 930 photos/s, with the GPU at 97%), and scoring val about 7 s.

In [ ]:
%load_ext autoreload
%autoreload 2

import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from omegaconf import OmegaConf
from torch import nn
from torch.utils.data import DataLoader

from face_verif_gen import metrics
from face_verif_gen.data import FaceDataset
from face_verif_gen.heads import ArcFaceHead, SoftmaxHead, cross_entropy_loss
from face_verif_gen.model import IResNet, preprocess
from face_verif_gen.train import train_step
from face_verif_gen.verify import bin_edges, embed, pair_histograms, pick_faces

REPO = Path.cwd().parent
CASIA = OmegaConf.load(REPO / "configs" / "data" / "casia.yaml")
LAB = REPO / "docs" / "lab"  # notes and plots, committed
RUNS = REPO / "outputs" / "lab"  # models and traces, gitignored
device = torch.device("cuda")


def casia(split, people=None):
    """One split of CASIA, read from the decoded array; people keeps only its first N people."""
    d = CASIA
    return FaceDataset(
        REPO / d.manifest, REPO / d.root, REPO / d.split_file, split, REPO / d.array, people
    )


def in_memory(dataset, num_workers=8):
    """Every image in a dataset as one uint8 tensor in RAM, and the labels."""
    loader = DataLoader(dataset, batch_size=1024, num_workers=num_workers)
    images, labels = zip(*loader, strict=True)
    return torch.cat(images), torch.cat(labels)


def batches(images, labels, batch_size=512):
    """One epoch of shuffled batches, copied to the GPU; the last part-batch is dropped."""
    order = torch.randperm(len(images))
    for start in range(0, len(order) - batch_size + 1, batch_size):
        i = order[start : start + batch_size]
        yield images[i].to(device), labels[i].to(device)


def val_tar(model, fars=(1e-3, 1e-4), degrade=None):
    """TAR at each FAR on val: 500 people, up to 20 photos each. Puts the model in eval mode.

    degrade: applied to each uint8 batch on the GPU first.
    """
    model.eval()
    emb = embed(model, val, val_faces, device, num_workers=8, degrade=degrade)
    genuine, impostor = pair_histograms(emb, val_labels)
    return {far: metrics.tar_at_far(EDGES, EDGES, far, genuine, impostor)[0] for far in fars}


def save(fig, name):
    """A lab plot, as docs/lab/<name>.png."""
    LAB.mkdir(exist_ok=True)
    fig.savefig(LAB / f"{name}.png", dpi=150, bbox_inches="tight")

In [ ]:
train = casia("train", people=1000)
train_images, train_labels = in_memory(train)  # 3.1 GB: too much to keep on the GPU as well
print(f"{len(train):,} photos of {train.num_classes:,} people")

val = casia("val")
val_faces = pick_faces(val.labels, per_identity=20, seed=0)
val_labels = torch.as_tensor(val.labels[val_faces], device=device)
EDGES = bin_edges()

## Baseline

I train softmax for 5 epochs with SGD at AdaFace's settings, a constant learning rate and M2's
`train_step`. It shows what a normal run looks like, and I save it to `outputs/lab/baseline.pt` for
the labs that need a trained model.

In [ ]:
torch.manual_seed(0)
model = IResNet().to(device)
head = SoftmaxHead(IResNet.embedding_dim, train.num_classes).to(device)
optimiser = torch.optim.SGD(
    [*model.parameters(), *head.parameters()], lr=0.1, momentum=0.9, weight_decay=1e-4
)

losses, tars = [], []
for epoch in range(5):
    start = time.perf_counter()
    for images, labels in batches(train_images, train_labels):
        losses.append(train_step(model, head, optimiser, images, labels))
    tars.append(val_tar(model))
    print(f"epoch {epoch}: {time.perf_counter() - start:.0f} s, loss {losses[-1]:.2f}, {tars[-1]}")

RUNS.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), RUNS / "baseline.pt")

fig, ax = plt.subplots()
ax.plot(losses)
ax.set(xlabel="step", ylabel="training loss", title="Baseline");

## L1. Learning rate (C3)

I run a range test to pick a learning rate, then train at the pick, 10× lower and 10× higher, to
see what too low and too high look like.

The range test is one epoch from a fresh model, with the rate rising geometrically from 1e-4 to 10.
The three runs use constant rates, so I check the scheduler's rate in L2, where warmup needs one.

In [ ]:
def softmax_sgd(lr, seed=0):
    """A fresh IResNet-18 and softmax head, and SGD at AdaFace's settings but with rate lr."""
    torch.manual_seed(seed)
    model = IResNet().to(device)
    head = SoftmaxHead(IResNet.embedding_dim, train.num_classes).to(device)
    optimiser = torch.optim.SGD(
        [*model.parameters(), *head.parameters()], lr=lr, momentum=0.9, weight_decay=1e-4
    )
    return model, head, optimiser


def range_test(lo=1e-4, hi=10.0, batch_size=512):
    """One epoch from a fresh model, the rate rising geometrically from lo to hi, one per step.

    Stops early once the loss is not finite or is 4x its lowest. Returns each step's rate and
    loss.
    """
    model, head, optimiser = softmax_sgd(lo)
    steps = len(train_images) // batch_size
    scheduler = torch.optim.lr_scheduler.ExponentialLR(
        # hi/lo = ratio from start to end
        # steps-1 = number of steps to get there
        optimiser,
        gamma=(hi / lo) ** (1 / (steps - 1)),
    )
    rates, losses = [], []
    for images, labels in batches(train_images, train_labels, batch_size):
        rates.append(optimiser.param_groups[0]["lr"])  # the rate this step uses
        losses.append(train_step(model, head, optimiser, images, labels))
        if not np.isfinite(losses[-1]) or losses[-1] > 4 * min(losses):
            break
        scheduler.step()
    return np.array(rates), np.array(losses)


rates, losses = range_test()
lowest = np.nanargmin(losses)  # the last loss can be NaN
pick = rates[lowest] / 10
print(f"lowest loss {losses[lowest]:.2f} at rate {rates[lowest]:.3g}: pick about {pick:.2g}")

fig, ax = plt.subplots()
ax.semilogx(rates, losses)
ax.axvline(pick, color="grey", linestyle="--", label=f"pick: a tenth of {rates[lowest]:.2g}")
ax.set(
    xlabel="learning rate",
    ylabel="training loss",
    ylim=(0, 4 * losses[lowest]),  # the stopping point; a diverged loss would flatten the rest
    title="L1: learning-rate range test",
)
ax.legend()
save(fig, "L1-range")

In [ ]:
def train_at(lr, epochs=5):
    """A fresh model trained at a constant rate lr: each step's loss, and val TAR after each epoch.

    Stops at the first loss that isn't finite, since every step after it would be NaN too.
    """
    model, head, optimiser = softmax_sgd(lr)
    losses, tars = [], []
    for epoch in range(epochs):
        start = time.perf_counter()
        for images, labels in batches(train_images, train_labels):
            losses.append(train_step(model, head, optimiser, images, labels))
            if not np.isfinite(losses[-1]):
                print(f"lr {lr:g}: loss not finite at step {len(losses) - 1}")
                return losses, tars
        tars.append(val_tar(model))
        took = time.perf_counter() - start
        print(f"lr {lr:g}, epoch {epoch}: {took:.0f} s, loss {losses[-1]:.2f}, {tars[-1]}")
    return losses, tars


PICK = 0.05  # the middle of the range test's steepest stretch, about 0.01 to 0.1
EPOCHS = 5
l1 = {lr: train_at(lr, EPOCHS) for lr in (PICK / 10, PICK, PICK * 10)}

fig, (loss_ax, tar_ax) = plt.subplots(1, 2, figsize=(11, 4))
for lr, (run_losses, run_tars) in l1.items():
    loss_ax.plot(run_losses, label=f"{lr:g}")
    tar_ax.plot(
        range(1, len(run_tars) + 1), [tar[1e-3] for tar in run_tars], marker="o", label=f"{lr:g}"
    )
loss_ax.set(
    xlabel="step",
    ylabel="training loss",
    ylim=(0, 1.5 * np.log(train.num_classes)),  # a model guessing at random scores ln(classes)
)
tar_ax.set(xlabel="epoch", ylabel="val TAR at FAR 1e-3", xticks=range(1, EPOCHS + 1))
loss_ax.legend(title="learning rate")
fig.suptitle("L1: training at a tenth of, at, and 10x the picked rate")
save(fig, "L1-runs")

## L2. Margin and warmup (D5, C3)

I train ArcFace (m = 0.5) with and without warmup, to reproduce the collapse from i_use_arc_btw and
see what warmup changes.

The collapse in i_use_arc_btw moved every face one shared way, which a final BatchNorm's learnable
shift makes easy: in training, each batch's embeddings average exactly that shift. AdaFace's
IResNet has no shift (`affine=False`), so I do 4 runs, every combination of: the shift on or off,
and one epoch of linear warmup or none, all at L1's rate. The first cell plots the planned
schedules, to check the warmup reaches the full rate at the right step.

In [ ]:
M = 0.5
WARMUP = len(train_images) // 512  # one epoch of steps
STEPS = EPOCHS * WARMUP


def schedule(optimiser, warmup_steps):
    """The rate rising linearly from 1% of the optimiser's over warmup_steps, then held there."""
    if warmup_steps == 0:
        return torch.optim.lr_scheduler.ConstantLR(optimiser, factor=1.0)
    return torch.optim.lr_scheduler.LinearLR(optimiser, start_factor=0.01, total_iters=warmup_steps)


def planned_rates(warmup_steps, steps=STEPS):
    """The rate schedule() gives at each step, read from a stand-in optimiser."""
    optimiser = torch.optim.SGD([torch.zeros(1, requires_grad=True)], lr=PICK)
    scheduler = schedule(optimiser, warmup_steps)
    planned = []
    for _ in range(steps):
        planned.append(optimiser.param_groups[0]["lr"])
        optimiser.step()  # no gradients, so nothing changes; without it the scheduler warns
        scheduler.step()
    return planned


planned = planned_rates(WARMUP)
print(
    f"rate at steps 0, {WARMUP - 1} and {WARMUP}: {planned[0]:.4g}, {planned[WARMUP - 1]:.4g}, "
    f"{planned[WARMUP]:.4g}"
)

fig, ax = plt.subplots()
for warmup_steps in (0, WARMUP):
    ax.plot(planned_rates(warmup_steps), label=f"warmup over {warmup_steps} steps")
ax.axvline(WARMUP, color="grey", linestyle=":")
ax.set(xlabel="step", ylabel="learning rate", title="L2: planned schedules")
ax.legend();

In [ ]:
# 75.7 degrees. When a face is further than this from every identity, moving it further from all
# of them at once lowers the loss.
TRAP = np.degrees((np.pi - M) / 2)


def arcface_sgd(lr, shift=False, seed=0):
    """A fresh IResNet-18, ArcFace head (s = 64, m = M) and SGD as AdaFace's, but at rate lr.

    shift: give the final BatchNorm a learnable scale and shift, as in i_use_arc_btw's neck.
    AdaFace's has neither, so in training each batch's embeddings average exactly zero.
    """
    torch.manual_seed(seed)
    model = IResNet()
    if shift:
        assert isinstance(model.output[-1], nn.BatchNorm1d)
        model.output[-1] = nn.BatchNorm1d(IResNet.embedding_dim)
    model = model.to(device)
    head = ArcFaceHead(IResNet.embedding_dim, train.num_classes, s=64.0, m=M).to(device)
    optimiser = torch.optim.SGD(
        [*model.parameters(), *head.parameters()], lr=lr, momentum=0.9, weight_decay=1e-4
    )
    return model, head, optimiser


def arcface_step(model, head, optimiser, images, labels):
    """train_step, also returning two signs of the trap, measured on the batch before the update.

    theta_y: the average angle, in degrees, between a face and its own identity's row.
    face_face: the average cosine between faces of different people; 0 is spread out, 1 collapsed.
    """
    model.train()
    with torch.autocast(images.device.type, dtype=torch.bfloat16):
        embeddings = model(preprocess(images))
    embeddings = embeddings.float()
    loss = cross_entropy_loss(head(embeddings, labels), labels)
    with torch.no_grad():
        faces = F.normalize(embeddings, dim=1)
        rows = F.normalize(head.fc.weight, dim=1)[labels]
        theta_y = torch.rad2deg(torch.acos((faces * rows).sum(1).clamp(-1, 1))).mean()
        different = labels[:, None] != labels[None, :]
        face_face = (faces @ faces.T)[different].mean()
    optimiser.zero_grad()
    loss.backward()
    optimiser.step()
    return loss.item(), theta_y.item(), face_face.item()


def train_arcface(shift, warmup_steps, epochs=EPOCHS):
    """ArcFace at rate PICK from a fresh model, warming up over warmup_steps.

    Returns each step's (loss, theta_y, face_face) as the rows of an array, and val TAR after each
    epoch. Stops at the first loss that isn't finite.
    """
    model, head, optimiser = arcface_sgd(PICK, shift)
    scheduler = schedule(optimiser, warmup_steps)
    steps, tars = [], []
    for epoch in range(epochs):
        start = time.perf_counter()
        for images, labels in batches(train_images, train_labels):
            steps.append(arcface_step(model, head, optimiser, images, labels))
            if not np.isfinite(steps[-1][0]):
                print(f"loss not finite at step {len(steps) - 1}")
                return np.array(steps), tars
            scheduler.step()
        tars.append(val_tar(model))
        loss, theta_y, face_face = steps[-1]
        print(
            f"epoch {epoch}: {time.perf_counter() - start:.0f} s, loss {loss:.2f}, "
            f"theta_y {theta_y:.1f}, face-face {np.degrees(np.arccos(face_face)):.1f}, {tars[-1]}"
        )
    return np.array(steps), tars


l2 = {}
for shift in (False, True):
    for warmup_steps in (0, WARMUP):
        print(f"shift {shift}, warmup over {warmup_steps} steps")
        l2[shift, warmup_steps] = train_arcface(shift, warmup_steps)

In [ ]:
fig, ((loss_ax, theta_ax), (face_ax, tar_ax)) = plt.subplots(
    2, 2, figsize=(11, 8), layout="constrained"
)
for colour, ((shift, warmup_steps), (run_steps, run_tars)) in enumerate(l2.items()):
    label = f"{'shift' if shift else 'no shift'}, {'warmup' if warmup_steps else 'no warmup'}"
    style = {"color": f"C{colour}", "linewidth": 1}
    loss_ax.plot(run_steps[:, 0], label=label, **style)
    theta_ax.plot(run_steps[:, 1], **style)
    face_ax.plot(np.degrees(np.arccos(run_steps[:, 2])), **style)  # the average cosine's angle
    epochs = range(1, len(run_tars) + 1)
    tar_ax.plot(epochs, [tar[1e-3] for tar in run_tars], marker="o", **style)
theta_ax.axhline(
    TRAP,
    color="grey",
    linestyle=":",
    label=f"{TRAP:.1f}°: above it, moving away from\nall identities lowers the loss",
)
loss_ax.set(xlabel="step", ylabel="training loss")
theta_ax.set(xlabel="step", ylabel="angle to own identity (degrees)")
face_ax.set(xlabel="step", ylabel="angle between different people (degrees)")
tar_ax.set(xlabel="epoch", ylabel="val TAR at FAR 1e-3", xticks=range(1, EPOCHS + 1))
theta_ax.legend(loc="center right")
fig.legend(*loss_ax.get_legend_handles_labels(), loc="outside lower center", ncols=4)
fig.suptitle(f"L2: ArcFace, m = {M}, at rate {PICK}")
save(fig, "L2")

## L3. Optimisers (C2, B8)

I compare SGD with momentum against AdamW, each at its best rate, with weight decay on and off.

SGD uses L1's rate. For AdamW I run its own range test first, and check its pick against the plot
before the runs. Weight decay "on" is AdaFace's 1e-4 for SGD and a common 0.05 for AdamW.

In [ ]:
WEIGHT_DECAY = {"sgd": 1e-4, "adamw": 0.05}  # "on": AdaFace's for SGD, a common value for AdamW


def softmax_with(optimiser, lr, weight_decay, seed=0):
    """A fresh IResNet-18 and softmax head, and an optimiser over both at rate lr.

    optimiser: "sgd", with momentum 0.9 as AdaFace's, or "adamw", with PyTorch's other defaults.
    """
    torch.manual_seed(seed)
    model = IResNet().to(device)
    head = SoftmaxHead(IResNet.embedding_dim, train.num_classes).to(device)
    params = [*model.parameters(), *head.parameters()]
    if optimiser == "sgd":
        opt = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    elif optimiser == "adamw":
        opt = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    else:
        raise ValueError(f"unknown optimiser {optimiser!r}")
    return model, head, opt


def range_test_with(optimiser, weight_decay, lo, hi, batch_size=512):
    """L1's range test, for any optimiser softmax_with() builds: each step's rate and loss."""
    model, head, opt = softmax_with(optimiser, lo, weight_decay)
    steps = len(train_images) // batch_size
    scheduler = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=(hi / lo) ** (1 / (steps - 1)))
    rates, losses = [], []
    for images, labels in batches(train_images, train_labels, batch_size):
        rates.append(opt.param_groups[0]["lr"])  # the rate this step uses
        losses.append(train_step(model, head, opt, images, labels))
        if not np.isfinite(losses[-1]) or losses[-1] > 4 * min(losses):
            break
        scheduler.step()
    return np.array(rates), np.array(losses)


adamw_rates, adamw_losses = range_test_with("adamw", WEIGHT_DECAY["adamw"], lo=1e-6, hi=1.0)
adamw_lowest = np.nanargmin(adamw_losses)  # the last loss can be NaN
adamw_pick = adamw_rates[adamw_lowest] / 10
print(
    f"AdamW: lowest loss {adamw_losses[adamw_lowest]:.2f} at rate "
    f"{adamw_rates[adamw_lowest]:.3g}: a tenth is {adamw_pick:.2g}"
)

fig, ax = plt.subplots()
ax.semilogx(adamw_rates, adamw_losses)
ax.axvline(
    adamw_pick,
    color="grey",
    linestyle="--",
    label=f"a tenth of {adamw_rates[adamw_lowest]:.2g}",
)
ax.set(
    xlabel="learning rate",
    ylabel="training loss",
    ylim=(0, 4 * adamw_losses[adamw_lowest]),  # the stopping point
    title="L3: AdamW range test",
)
ax.legend()
save(fig, "L3-range")

In [ ]:
SGD_PICK = PICK  # L1's
ADAMW_PICK = 0.002  # adamw_pick  # the range test's rule; replace it if the plot disagrees


def train_with(optimiser, lr, weight_decay, epochs=EPOCHS):
    """L1's train_at, for any optimiser softmax_with() builds.

    Returns each step's loss, and val TAR after each epoch. Stops at the first loss that isn't
    finite.
    """
    model, head, opt = softmax_with(optimiser, lr, weight_decay)
    losses, tars = [], []
    for epoch in range(epochs):
        start = time.perf_counter()
        for images, labels in batches(train_images, train_labels):
            losses.append(train_step(model, head, opt, images, labels))
            if not np.isfinite(losses[-1]):
                print(f"loss not finite at step {len(losses) - 1}")
                return losses, tars
        tars.append(val_tar(model))
        took = time.perf_counter() - start
        print(f"epoch {epoch}: {took:.0f} s, loss {losses[-1]:.2f}, {tars[-1]}")
    return losses, tars


l3 = {}
for optimiser, lr in (("sgd", SGD_PICK), ("adamw", ADAMW_PICK)):
    for weight_decay in (WEIGHT_DECAY[optimiser], 0.0):
        print(f"{optimiser} at rate {lr:.2g}, weight decay {weight_decay:g}")
        l3[optimiser, weight_decay] = train_with(optimiser, lr, weight_decay)

In [ ]:
fig, (loss_ax, tar_ax) = plt.subplots(1, 2, figsize=(11, 4.5), layout="constrained")
for colour, ((optimiser, weight_decay), (run_losses, run_tars)) in enumerate(l3.items()):
    label = f"{optimiser}, weight decay {weight_decay:g}"
    style = {"color": f"C{colour}", "linewidth": 1}
    loss_ax.plot(run_losses, label=label, **style)
    epochs = range(1, len(run_tars) + 1)
    tar_ax.plot(epochs, [tar[1e-3] for tar in run_tars], marker="o", **style)
loss_ax.set(
    xlabel="step",
    ylabel="training loss",
    ylim=(0, 1.5 * np.log(train.num_classes)),  # a model guessing at random scores ln(classes)
)
tar_ax.set(xlabel="epoch", ylabel="val TAR at FAR 1e-3", xticks=range(1, EPOCHS + 1))
fig.legend(*loss_ax.get_legend_handles_labels(), loc="outside lower center", ncols=4)
fig.suptitle(f"L3: SGD at {SGD_PICK:g} and AdamW at {ADAMW_PICK:.2g}, weight decay on and off")
save(fig, "L3")

## L4. Gradient checks (C7, C15)

I log how far each step moves each layer, to see whether that shows a run failing before its val
score does. Then I try gradient clipping in place of warmup.

How far a step moves a layer is the change to its weights over their size. Every layer logged is
followed by a BatchNorm, or normalised in the head, so only the weights' direction matters, and
this is how far the step turned them. A common rule of thumb is about 1e-3 a step: much more and
the layer changes too fast to settle, much less and it has stopped learning.

I do three runs of L2's ArcFace without the shift, at L1's rate: with warmup, which learned in L2;
without, which collapsed; and without but clipped. Clipping scales a step's gradient down to a cap
whenever it is bigger. The cap is the warmup run's usual gradient size after warmup.

In [ ]:
def tracked_weights(model, head):
    """Every conv and linear weight, input to output, by a short name.

    Each is followed by a BatchNorm, or normalised in the head, so only its direction matters.
    """
    names = {"stem.0": "stem", "output.3": "embedding", "fc": "head"}
    weights = {}
    for name, module in [*model.named_modules(), *head.named_modules()]:
        if isinstance(module, nn.Conv2d | nn.Linear):
            weights[names.get(name, name.removeprefix("body.").removesuffix(".0"))] = module.weight
    return weights


def checked_step(model, head, optimiser, weights, images, labels, clip=None):
    """L2's arcface_step without face_face, clipping the gradient to size clip if given.

    Returns the loss, theta_y, the gradient's size before clipping, and how far the step moved
    each of weights: the change's size over the weights' size.
    """
    model.train()
    with torch.autocast(images.device.type, dtype=torch.bfloat16):
        embeddings = model(preprocess(images))
    embeddings = embeddings.float()
    loss = cross_entropy_loss(head(embeddings, labels), labels)
    with torch.no_grad():
        faces = F.normalize(embeddings, dim=1)
        rows = F.normalize(head.fc.weight, dim=1)[labels]
        theta_y = torch.rad2deg(torch.acos((faces * rows).sum(1).clamp(-1, 1))).mean()
    optimiser.zero_grad()
    loss.backward()
    params = [p for group in optimiser.param_groups for p in group["params"]]
    # The size of the gradient over every parameter. With no clip the cap is infinite, so the
    # gradient is left as it is.
    grad_norm = torch.nn.utils.clip_grad_norm_(params, clip or float("inf"))
    before = [w.detach().clone() for w in weights]
    optimiser.step()
    with torch.no_grad():
        moved = torch.stack(
            [(w - b).norm() / b.norm() for w, b in zip(weights, before, strict=True)]
        )
    return loss.item(), theta_y.item(), grad_norm.item(), moved.cpu().numpy()


def columns(rows):
    """checked_step's results, one per step, as one array per measure."""
    loss, theta_y, grad_norm, moved = zip(*rows, strict=True)
    return {
        "loss": np.array(loss),
        "theta_y": np.array(theta_y),
        "grad_norm": np.array(grad_norm),
        "moved": np.stack(moved),  # one row per step, one column per tracked weight
    }


def train_checked(warmup_steps, clip=None, epochs=EPOCHS):
    """L2's ArcFace without the shift, at rate PICK, warming up over warmup_steps.

    Returns columns() of every step, and val TAR after each epoch. Stops at the first loss that
    isn't finite.
    """
    model, head, optimiser = arcface_sgd(PICK)
    weights = list(tracked_weights(model, head).values())
    scheduler = schedule(optimiser, warmup_steps)
    rows, tars = [], []
    for epoch in range(epochs):
        start = time.perf_counter()
        for images, labels in batches(train_images, train_labels):
            rows.append(checked_step(model, head, optimiser, weights, images, labels, clip))
            if not np.isfinite(rows[-1][0]):
                print(f"loss not finite at step {len(rows) - 1}")
                return columns(rows), tars
            scheduler.step()
        tars.append(val_tar(model))
        loss, theta_y, grad_norm, _ = rows[-1]
        print(
            f"epoch {epoch}: {time.perf_counter() - start:.0f} s, loss {loss:.2f}, "
            f"theta_y {theta_y:.1f}, gradient size {grad_norm:.3g}, {tars[-1]}"
        )
    return columns(rows), tars


LAYERS = list(tracked_weights(IResNet(), ArcFaceHead(IResNet.embedding_dim, train.num_classes)))

print("warmup")
l4 = {"warmup": train_checked(WARMUP)}
CLIP = float(np.median(l4["warmup"][0]["grad_norm"][WARMUP:]))  # its usual size after warmup
print("no warmup")
l4["no warmup"] = train_checked(0)
print(f"no warmup, clipped at {CLIP:.3g}")
l4["no warmup, clipped"] = train_checked(0, clip=CLIP)

In [ ]:
fig = plt.figure(figsize=(13, 9), layout="constrained")
top, bottom = fig.subfigures(2, 1, height_ratios=[1.3, 1])
maps = top.subplots(1, len(l4), sharey=True)
grad_ax, theta_ax, tar_ax = bottom.subplots(1, 3)

# Colours centred on the rule of thumb, 1e-3 a step: red moves a layer more, blue less.
logs = np.concatenate([np.log10(run["moved"]).ravel() for run, _ in l4.values()])
spread = np.percentile(np.abs(logs[np.isfinite(logs)] + 3), 99)
norm = plt.Normalize(-3 - spread, -3 + spread)
for colour, (ax, (name, (run, run_tars))) in enumerate(zip(maps, l4.items(), strict=True)):
    image = ax.imshow(
        np.log10(run["moved"]).T,
        aspect="auto",
        origin="lower",
        cmap="RdBu_r",
        norm=norm,
        interpolation="nearest",
    )
    ax.set(title=name, xlabel="step", xlim=(-0.5, STEPS - 0.5))
    style = {"color": f"C{colour}", "linewidth": 1, "label": name}
    grad_ax.plot(run["grad_norm"], **style)
    theta_ax.plot(run["theta_y"], **style)
    epochs = range(1, len(run_tars) + 1)
    tar_ax.plot(epochs, [tar[1e-3] for tar in run_tars], marker="o", **style)
maps[0].set(yticks=range(len(LAYERS)), yticklabels=LAYERS, ylabel="layer, input at the bottom")
maps[0].tick_params(axis="y", labelsize=7)
bar = top.colorbar(image, ax=maps, label="each step's change to a layer's weights / their size")
ticks = np.arange(np.ceil(norm.vmin), np.floor(norm.vmax) + 1)
bar.set_ticks(ticks, labels=[f"1e{tick:.0f}" for tick in ticks])

cap = grad_ax.axhline(CLIP, color="grey", linestyle=":", label=f"clip at {CLIP:.3g}")
grad_ax.set(xlabel="step", ylabel="gradient size, before clipping", yscale="log")
grad_ax.legend(handles=[cap], loc="upper right")
trap = theta_ax.axhline(TRAP, color="grey", linestyle=":", label=f"{TRAP:.1f}°, as in L2")
theta_ax.set(xlabel="step", ylabel="angle to own identity (degrees)")
theta_ax.legend(handles=[trap], loc="upper right")
tar_ax.set(xlabel="epoch", ylabel="val TAR at FAR 1e-3", xticks=range(1, EPOCHS + 1))
bottom.legend(*tar_ax.get_legend_handles_labels(), loc="outside lower center", ncols=3)
fig.suptitle(f"L4: ArcFace at rate {PICK}, how far each step moves each layer")
save(fig, "L4")

## L5. Batch norm (C4, C16, F5)

I score a model left in train mode, then train with batches of 8 and of 256, and with 8s
accumulated up to 256, to see whether accumulation recovers the large-batch result.

In train mode, each BatchNorm normalises a batch with that batch's own mean and spread, so a
photo's embedding depends on the other photos in its batch; dropout is on too. In eval mode it uses
running averages kept during training. First I score the baseline on val four ways: in eval mode;
in train mode, with val in its usual order, each person's photos together; in train mode,
shuffled; and in eval mode again after one pass in train mode, which overwrites those running
averages with val's batches.

Then I train softmax for 2 epochs three ways: 256 photos a step; 8 a step; and 256 a step run
through the model in 32 pieces of 8, their gradients added up before one step. The rate is L1's
0.05 times the photos a step over 512. Each step averages the gradient over its photos, so 32
times fewer photos a step means 32 times more steps, and a 32 times smaller rate moves the weights
about as far per epoch. The pieces add up to exactly the 256 batch's gradient, except that each
BatchNorm normalises 8 photos at a time.

In [ ]:
from torch.utils.data import Subset

val_images, _ = in_memory(Subset(val, val_faces))  # in val_labels' order


def baseline():
    """The baseline model, freshly loaded from outputs/lab/baseline.pt."""
    model = IResNet().to(device)
    model.load_state_dict(torch.load(RUNS / "baseline.pt", weights_only=True))
    return model


@torch.no_grad()
def embed_train_mode(model, order, batch_size=256):
    """Unit-length embeddings of val_images[order], 256 to a batch, with the model in train mode.

    This is what embed() refuses to do.
    """
    model.train()
    chunks = [
        model(preprocess(images.to(device))) for images in val_images[order].split(batch_size)
    ]
    return F.normalize(torch.cat(chunks).float(), dim=1)


def tar_of(embeddings, labels, far=1e-3):
    """TAR at far over every pair of embeddings, as val_tar() scores them."""
    genuine, impostor = pair_histograms(embeddings, labels)
    return metrics.tar_at_far(EDGES, EDGES, far, genuine, impostor)[0]


in_order = torch.arange(len(val_faces))  # val_faces is sorted, so each person's photos are together
shuffled = torch.randperm(len(val_faces), generator=torch.Generator().manual_seed(0))
people = [len(val_labels[batch].unique()) for batch in in_order.split(256)]
print(f"in order, a batch of 256 holds {np.mean(people):.1f} people on average")

modes = {"eval mode": val_tar(baseline())[1e-3]}
for name, order in (("in order", in_order), ("shuffled", shuffled)):
    embeddings = embed_train_mode(baseline(), order)
    modes[f"train mode, {name}"] = tar_of(embeddings, val_labels[order])
model = baseline()
embed_train_mode(model, in_order)  # updates every BatchNorm's running averages with val's batches
modes["eval mode, after a pass in train mode"] = val_tar(model)[1e-3]
for name, tar in modes.items():
    print(f"{name}: TAR {tar:.3f}")

In [ ]:
def accumulated_step(model, head, optimiser, images, labels, pieces=1):
    """One step on a batch, run through the model in pieces of equal size; the batch's loss.

    Each piece's loss is divided by pieces, so the gradients add up to the whole batch's average,
    but each BatchNorm normalises one piece at a time. pieces=1 is train_step.
    """
    model.train()
    optimiser.zero_grad()
    loss = 0.0
    for piece, piece_labels in zip(images.chunk(pieces), labels.chunk(pieces), strict=True):
        with torch.autocast(images.device.type, dtype=torch.bfloat16):
            embeddings = model(preprocess(piece))
        piece_loss = cross_entropy_loss(head(embeddings.float(), piece_labels), piece_labels)
        (piece_loss / pieces).backward()
        loss += piece_loss.item() / pieces
    optimiser.step()
    return loss


def train_batched(batch_size, pieces=1, epochs=2):
    """Softmax from a fresh model, batch_size photos a step, each step run in pieces.

    The rate is L1's PICK scaled by batch_size / 512. Returns each step's loss and val TAR after
    each epoch. Stops at the first loss that isn't finite.
    """
    model, head, optimiser = softmax_sgd(PICK * batch_size / 512)
    losses, tars = [], []
    for epoch in range(epochs):
        start = time.perf_counter()
        for images, labels in batches(train_images, train_labels, batch_size):
            losses.append(accumulated_step(model, head, optimiser, images, labels, pieces))
            if not np.isfinite(losses[-1]):
                print(f"loss not finite at step {len(losses) - 1}")
                return np.array(losses), tars
        tars.append(val_tar(model))
        took = time.perf_counter() - start
        print(f"epoch {epoch}: {took:.0f} s, loss {losses[-1]:.2f}, {tars[-1]}")
    return np.array(losses), tars


l5 = {}
for name, batch_size, pieces in (
    ("256 a step", 256, 1),
    ("8 a step", 8, 1),
    ("256 a step, in 32 pieces of 8", 256, 32),
):
    print(f"{name}, rate {PICK * batch_size / 512:.2g}")
    l5[name] = batch_size, train_batched(batch_size, pieces)

In [ ]:
fig, (mode_ax, loss_ax, tar_ax) = plt.subplots(
    1, 3, figsize=(15, 4.5), layout="constrained", width_ratios=[1.2, 1, 1]
)
bars = mode_ax.barh(list(modes), list(modes.values()), color="C7")
mode_ax.bar_label(bars, fmt="%.3f", padding=3)
mode_ax.invert_yaxis()  # first condition at the top
mode_ax.set(xlabel="val TAR at FAR 1e-3", title="Baseline, scored four ways")
mode_ax.margins(x=0.15)

for colour, (name, (batch_size, (run_losses, run_tars))) in enumerate(l5.items()):
    style = {"color": f"C{colour}", "linewidth": 1, "label": name}
    per_step = 256 // batch_size  # steps per 256 photos, so every run is averaged the same way
    whole = len(run_losses) // per_step * per_step
    loss_ax.plot(
        256 * np.arange(1, whole // per_step + 1),
        run_losses[:whole].reshape(-1, per_step).mean(1),
        **style,
    )
    tar_ax.plot(range(1, len(run_tars) + 1), [tar[1e-3] for tar in run_tars], marker="o", **style)
loss_ax.set(
    xlabel="photos seen",
    ylabel="training loss, averaged over each 256 photos",
    ylim=(0, 1.5 * np.log(train.num_classes)),  # a model guessing at random scores ln(classes)
)
epochs = len(next(iter(l5.values()))[1][1])
tar_ax.set(xlabel="epoch", ylabel="val TAR at FAR 1e-3", xticks=range(1, epochs + 1))
tar_ax.legend()
fig.suptitle("L5: BatchNorm in train mode, and small batches")
save(fig, "L5")

## L6. Precision (C12, C14)

I remove the acos clamp and find the NaN with anomaly detection. Then I compare fp16 without loss
scaling, fp16 with it, and bf16, and measure speed and memory for fp32 against bf16.

ArcFace takes acos of the cosine between a face and its identity's row. acos of anything above 1
is NaN, and its slope is infinite at 1. I build that case on purpose: 512 faces, each placed
exactly on its identity's row. In exact arithmetic every cosine is 1; fp32 rounding puts some just
above. The head without the clamp is a copy of `ArcFaceHead` made here; heads.py is unchanged. I
run one forward and backward pass through each head, first plainly, then under anomaly detection.
Anomaly detection checks what each op's backward returns and stops at the first NaN; its warning
shows the line where that op ran in the forward pass.

fp16 has 5 bits of exponent and 10 of precision: the smallest value it holds is about 6e-8, and
values below about 6e-5 keep fewer bits. bf16 has fp32's 8 bits of exponent, so the same range as
fp32, but only 7 bits of precision. I train softmax for one epoch three ways: fp16, fp16 with loss
scaling, and bf16. Loss scaling multiplies the loss by a large number (65,536 to start) before the
backward pass, so small gradients stay inside fp16's range, then divides the gradients by it before
the step. If any gradient came out infinite, it skips that step and halves the number. Each step
logs the share of the model's gradient values that are exactly 0.

Last, I time 20 steps of 128 photos in fp32 and in bf16, reusing one batch already on the GPU, and
record each one's peak memory.

In [ ]:
import math


class UnclampedArcFace(ArcFaceHead):
    """ArcFaceHead with acos given the raw cosine, not one clamped inside (-1, 1)."""

    def forward(self, emb, labels):
        cos = F.linear(F.normalize(emb, dim=1), F.normalize(self.fc.weight, dim=1))
        cos_y = cos.gather(1, labels[:, None])
        theta_y = torch.acos(cos_y)  # NaN wherever cos_y is above 1
        target = torch.cos(theta_y + self.m)
        fallback = cos_y - self.m * math.sin(self.m)
        target = torch.where(theta_y + self.m < math.pi, target, fallback)
        return self.s * cos.scatter(1, labels[:, None], target)


def head_step(head, faces, labels, anomaly=False):
    """One forward and backward pass through head and the loss, under anomaly detection if asked.

    Returns the loss, the share of gradient values that are NaN (over the faces and the head's
    rows), and the first line of the error anomaly detection raised, if any. Its warning, printed
    below the cell, shows the forward line of the op whose backward returned NaN.
    """
    faces = faces.clone().requires_grad_()
    head.zero_grad()
    with torch.autograd.set_detect_anomaly(anomaly):
        loss = cross_entropy_loss(head(faces, labels), labels)
        try:
            loss.backward()
        except RuntimeError as error:
            return loss.item(), None, str(error).splitlines()[0]
    grads = torch.cat([faces.grad.flatten(), head.fc.weight.grad.flatten()])
    return loss.item(), grads.isnan().float().mean().item(), None


torch.manual_seed(0)
clamped = ArcFaceHead(IResNet.embedding_dim, train.num_classes, s=64.0, m=M).to(device)
unclamped = UnclampedArcFace(IResNet.embedding_dim, train.num_classes, s=64.0, m=M).to(device)
unclamped.load_state_dict(clamped.state_dict())  # the same identity rows
identities = torch.randperm(train.num_classes, device=device)[:512]
on_row = clamped.fc.weight.detach()[identities]  # each face exactly on its identity's row

with torch.no_grad():
    rows = F.normalize(clamped.fc.weight, dim=1)
    cos_y = F.linear(F.normalize(on_row, dim=1), rows).gather(1, identities[:, None])
print(
    f"cos_y, 1 in exact arithmetic: {(cos_y > 1).sum()} above 1, {(cos_y == 1).sum()} exactly 1, "
    f"{(cos_y < 1).sum()} below 1"
)
for name, head in (("clamped", clamped), ("unclamped", unclamped)):
    loss, nan_share, _ = head_step(head, on_row, identities)
    print(f"{name}: loss {loss:.3g}, {nan_share:.1%} of gradient values NaN")
    _, _, error = head_step(head, on_row, identities, anomaly=True)
    print(f"{name}, under anomaly detection: {error or 'no error'}")

In [ ]:
def precision_step(model, head, optimiser, images, labels, dtype=None, scaler=None):
    """train_step under autocast at dtype, or in plain fp32 if dtype is None, scaling the loss
    with scaler if given.

    Returns the loss, and the share of the model's gradient values that are exactly 0, after
    unscaling.
    """
    model.train()
    with torch.autocast(device.type, dtype=dtype, enabled=dtype is not None):
        embeddings = model(preprocess(images))
    loss = cross_entropy_loss(head(embeddings.float(), labels), labels)
    optimiser.zero_grad()
    if scaler is None:
        loss.backward()
    else:
        scaler.scale(loss).backward()  # the loss times the scale, so every gradient is too
        scaler.unscale_(optimiser)  # divides the gradients by the scale, in fp32
    with torch.no_grad():
        grads = [p.grad for p in model.parameters()]
        zero = sum((g == 0).sum() for g in grads) / sum(g.numel() for g in grads)
    if scaler is None:
        optimiser.step()
    else:
        scaler.step(optimiser)  # skipped if any gradient is infinite or NaN
        scaler.update()  # halves the scale after a skipped step
    return loss.item(), zero.item()


PRECISIONS = {  # name: autocast dtype, loss scaling
    "fp16": (torch.float16, False),
    "fp16, loss scaling": (torch.float16, True),
    "bf16": (torch.bfloat16, False),
}


def train_precision(dtype, scaled, epochs=1):
    """Softmax at rate PICK from a fresh model, under autocast at dtype, loss scaled if scaled.

    Returns each step's (loss, share of zero gradient values) as the rows of an array; the loss
    scale before the first step and after each step (empty if not scaled); and val TAR after each
    epoch. Stops at the first loss that isn't finite.
    """
    model, head, optimiser = softmax_sgd(PICK)
    scaler = torch.amp.GradScaler(device.type) if scaled else None
    steps, tars = [], []
    scales = [scaler.get_scale()] if scaled else []
    for epoch in range(epochs):
        start = time.perf_counter()
        for images, labels in batches(train_images, train_labels):
            steps.append(precision_step(model, head, optimiser, images, labels, dtype, scaler))
            if scaled:
                scales.append(scaler.get_scale())
            if not np.isfinite(steps[-1][0]):
                print(f"loss not finite at step {len(steps) - 1}")
                return np.array(steps), np.array(scales), tars
        tars.append(val_tar(model))
        loss, zero = steps[-1]
        took = time.perf_counter() - start
        print(
            f"epoch {epoch}: {took:.0f} s, loss {loss:.2f}, zero gradients {zero:.1%}, {tars[-1]}"
        )
    return np.array(steps), np.array(scales), tars


l6 = {}
for name, (dtype, scaled) in PRECISIONS.items():
    print(name)
    l6[name] = train_precision(dtype, scaled)
scales = l6["fp16, loss scaling"][1]
print(
    f"loss scale: {scales[0]:g} at the start, {scales[-1]:g} at the end, "
    f"{(np.diff(scales) < 0).sum()} steps skipped"
)

In [ ]:
def step_cost(dtype, batch_size=128, steps=20, warmup=3):
    """Seconds per training step under autocast at dtype (None: plain fp32), and peak memory in GB.

    One batch, already on the GPU, is reused, so no data loading is timed. The first warmup steps
    are left out: they include cuDNN choosing its algorithms. Memory counts only what this run
    adds to what the notebook already holds.
    """
    torch.cuda.empty_cache()
    held = torch.cuda.memory_allocated()
    model, head, optimiser = softmax_sgd(PICK)
    images, labels = train_images[:batch_size].to(device), train_labels[:batch_size].to(device)
    torch.cuda.reset_peak_memory_stats()
    for step in range(warmup + steps):
        if step == warmup:
            torch.cuda.synchronize()
            start = time.perf_counter()
        precision_step(model, head, optimiser, images, labels, dtype)
    torch.cuda.synchronize()
    took = (time.perf_counter() - start) / steps
    return took, (torch.cuda.max_memory_allocated() - held) / 1e9


print(f"fp32 convolutions may run in TF32: {torch.backends.cudnn.allow_tf32}")
costs = {"fp32": step_cost(None), "bf16": step_cost(torch.bfloat16)}
for name, (took, peak) in costs.items():
    print(f"{name}: {1000 * took:.0f} ms a step of 128 photos, peak memory {peak:.2f} GB")

In [ ]:
from matplotlib.ticker import PercentFormatter

fig, (loss_ax, zero_ax, cost_ax) = plt.subplots(
    1, 3, figsize=(16, 4.5), layout="constrained", width_ratios=[1, 1, 0.8]
)
for colour, (name, (steps, _, tars)) in enumerate(l6.items()):
    label = f"{name}: TAR {tars[-1][1e-3]:.3f}" if tars else f"{name}: stopped"
    style = {"color": f"C{colour}", "linewidth": 1, "label": label}
    loss_ax.plot(steps[:, 0], **style)
    zero_ax.plot(steps[:, 1], **style)
loss_ax.set(
    xlabel="step",
    ylabel="training loss",
    ylim=(0, 1.5 * np.log(train.num_classes)),  # a model guessing at random scores ln(classes)
)
loss_ax.legend(title="val TAR at FAR 1e-3 after the epoch")
zero_ax.set(xlabel="step", ylabel="model's gradient values that are exactly 0")
zero_ax.yaxis.set_major_formatter(PercentFormatter(1))

bars = cost_ax.barh(list(costs), [1000 * took for took, _ in costs.values()], color="C7")
cost_ax.bar_label(
    bars,
    labels=[f"{1000 * took:.0f} ms, {peak:.1f} GB" for took, peak in costs.values()],
    padding=3,
)
cost_ax.invert_yaxis()  # fp32 at the top
cost_ax.margins(x=0.6)
cost_ax.set(xlabel="ms a step of 128 photos", title="Speed and peak memory")
fig.suptitle("L6: precision")
save(fig, "L6")

## L7. Overfitting (B1, B2, C5, C11)

I plot learning curves on 200 people, with and without weight decay, dropout and flips.

The first 200 training people have 23,322 photos. I hold out a random 10% of each person's photos,
which the model never trains on. After each epoch I measure the loss in eval mode on the held out
photos and on the same number of photos it trains on, and val TAR on new people. Overfitting shows
as a gap: the loss on photos it trains on keeps falling while the held out loss stops falling or
rises.

I do five runs of softmax at L1's rate, 8 epochs each, about 4 minutes a run: no regularisation;
weight decay only (1e-4, AdaFace's); dropout only (0.4, IResNet's); flips only (a random half of
each batch mirrored left to right); and all three. Unlike the earlier labs, "no regularisation"
turns both dropout and weight decay off.

In [ ]:
def hold_out(labels, share=0.1, seed=0):
    """A random share of each person's photos, at least one each: a boolean mask over labels."""
    rng = np.random.default_rng(seed)
    held = np.zeros(len(labels), dtype=bool)
    for person in np.unique(labels):
        photos = np.flatnonzero(labels == person)
        held[rng.choice(photos, size=max(1, round(share * len(photos))), replace=False)] = True
    return torch.from_numpy(held)


small = casia("train", people=200)
small_images, small_labels = in_memory(small)
held = hold_out(small_labels.numpy())
fit_images, fit_labels = small_images[~held], small_labels[~held]
held_images, held_labels = small_images[held], small_labels[held]
# As many photos it trains on as are held out, to score the same way.
order = torch.randperm(len(fit_images), generator=torch.Generator().manual_seed(0))
seen = order[: len(held_images)]
seen_images, seen_labels = fit_images[seen], fit_labels[seen]
print(
    f"{len(small):,} photos of {small.num_classes} people: {len(fit_images):,} to train on, "
    f"{len(held_images):,} held out"
)


@torch.no_grad()
def eval_loss(model, head, images, labels, batch_size=512):
    """The mean softmax loss over images in eval mode: no dropout, BatchNorm's running averages."""
    model.eval()
    total = 0.0
    for batch, batch_labels in zip(images.split(batch_size), labels.split(batch_size), strict=True):
        batch, batch_labels = batch.to(device), batch_labels.to(device)
        with torch.autocast(device.type, dtype=torch.bfloat16):
            embeddings = model(preprocess(batch))
        loss = cross_entropy_loss(head(embeddings.float(), batch_labels), batch_labels)
        total += loss.item() * len(batch)
    return total / len(images)


def regularised_step(model, head, optimiser, images, labels, flips):
    """train_step, first mirroring a random half of the photos left to right if flips."""
    if flips:
        mirror = torch.rand(len(images), device=images.device) < 0.5
        images = torch.where(mirror[:, None, None, None], images.flip(-1), images)
    return train_step(model, head, optimiser, images, labels)


def learning_curve(weight_decay, dropout, flips, epochs=8, seed=0):
    """Softmax at rate PICK on the 200 people's photos not held out, from a fresh model.

    Returns one row per epoch: the eval-mode loss on photos it trains on, the same on held out
    photos, and val TAR at FAR 1e-3. Stops at the first loss that isn't finite.
    """
    torch.manual_seed(seed)
    model = IResNet().to(device)
    model.output[1].p = dropout
    head = SoftmaxHead(IResNet.embedding_dim, small.num_classes).to(device)
    optimiser = torch.optim.SGD(
        [*model.parameters(), *head.parameters()], lr=PICK, momentum=0.9, weight_decay=weight_decay
    )
    curve = []
    for epoch in range(epochs):
        start = time.perf_counter()
        for images, labels in batches(fit_images, fit_labels):
            loss = regularised_step(model, head, optimiser, images, labels, flips)
            if not np.isfinite(loss):
                print(f"loss not finite in epoch {epoch}")
                return np.array(curve)
        seen_loss = eval_loss(model, head, seen_images, seen_labels)
        held_loss = eval_loss(model, head, held_images, held_labels)
        curve.append((seen_loss, held_loss, val_tar(model)[1e-3]))
        print(
            f"epoch {epoch}: {time.perf_counter() - start:.0f} s, loss {seen_loss:.2f} on photos "
            f"it trains on, {held_loss:.2f} held out, TAR {curve[-1][2]:.3f}"
        )
    return np.array(curve)


REGULARISATION = {  # name: weight decay, dropout, flips
    "none": (0.0, 0.0, False),
    "weight decay": (1e-4, 0.0, False),
    "dropout": (0.0, 0.4, False),
    "flips": (0.0, 0.0, True),
    "all three": (1e-4, 0.4, True),
}
l7 = {}
for name, (weight_decay, dropout, flips) in REGULARISATION.items():
    print(name)
    l7[name] = learning_curve(weight_decay, dropout, flips)

In [ ]:
fig, (loss_ax, tar_ax) = plt.subplots(1, 2, figsize=(12, 4.5), layout="constrained")
for colour, (name, curve) in enumerate(l7.items()):
    epochs = range(1, len(curve) + 1)
    loss_ax.plot(epochs, curve[:, 0], color=f"C{colour}", linewidth=1, label=name)
    loss_ax.plot(epochs, curve[:, 1], color=f"C{colour}", linewidth=1, linestyle="--")
    tar_ax.plot(epochs, curve[:, 2], color=f"C{colour}", marker="o", label=name)
# Entries for the two line styles, drawn with no data.
loss_ax.plot([], [], color="grey", label="photos it trains on")
loss_ax.plot([], [], color="grey", linestyle="--", label="held out photos")
loss_ax.set(xlabel="epoch", ylabel="loss in eval mode")
loss_ax.set_ylim(bottom=0)
loss_ax.legend()
tar_ax.set(xlabel="epoch", ylabel="val TAR at FAR 1e-3")
tar_ax.legend()
fig.suptitle("L7: 200 people, with and without regularisation")
save(fig, "L7")

## L8. Leakage (B3)

The split keeps each person in one set, so no name is in both train and val. Here I check for the
leakage the split can't catch: the same photo filed under two people, or twice under one. I report
counts and scores only, never the photos.

First I look for exact copies. I hash the pixels of every val and train photo (blake2b, 16 bytes),
using all of train, not the lab's 1,000 people, and reading the decoded array in file order; test
is not read. Equal hashes mean identical pixels, even when the PNG files differ. I count val photos
with an exact copy under the same val person, under another val person, and in train, and how many
of those are among the photos `val_tar` scores.

Then I look for near copies, such as a recompressed or re-cropped photo. I embed every val and
train photo with the baseline (about 445,000 photos, a few minutes) and find each val photo's
highest cosine to any train photo. With no copy in train, that is its best match among 422,091
photos of other people, unless the same person is also filed in train under another identity. An
exact copy scores 1. The plot puts these next to every val genuine and impostor pair, where copies
show as a separate bump near 1. Copies inside val show there too: genuine pairs near 1 are likely
one photo filed twice under one person, and impostor pairs near 1 one photo filed under two people.

In [ ]:
import hashlib

import pandas as pd


def pixel_hashes(dataset, chunk=4096):
    """A 16-byte hash of each photo's pixels, in the dataset's order, from the decoded array.

    The dataset's rows are in file order, so each chunk is read as one long run.
    """
    images = np.load(REPO / CASIA.array, mmap_mode="r")
    hashes = []
    for start in range(0, len(dataset.rows), chunk):
        block = images[dataset.rows[start : start + chunk]]  # reads these rows into RAM
        hashes += [hashlib.blake2b(image, digest_size=16).digest() for image in block]
    return np.array(hashes)


start = time.perf_counter()
all_train = casia("train")  # every training person, not the lab's 1,000
train_hashes = pixel_hashes(all_train)
val_hashes = pixel_hashes(val)
print(
    f"hashed {len(train_hashes) + len(val_hashes):,} photos in {time.perf_counter() - start:.0f} s"
)

val_copies = pd.DataFrame({"hash": val_hashes, "person": val.labels})
people_per_hash = val_copies.groupby("hash")["person"].nunique()
same_person = val_copies.duplicated(["hash", "person"], keep=False).to_numpy()
other_person = val_copies["hash"].isin(people_per_hash.index[people_per_hash > 1]).to_numpy()
exact = {
    "have a copy filed under the same val person": same_person,
    "have a copy filed under another val person": other_person,
    "have a copy in train": np.isin(val_hashes, train_hashes),
}
for name, copied in exact.items():
    print(
        f"{copied.sum():,} of {len(val):,} val photos {name}; "
        f"{copied[val_faces].sum()} of them among the {len(val_faces):,} val_tar scores"
    )

In [ ]:
model = baseline()
model.eval()
start = time.perf_counter()
val_embeddings = embed(model, val, range(len(val)), device, batch_size=512, num_workers=8)
train_embeddings = embed(
    model, all_train, range(len(all_train)), device, batch_size=512, num_workers=8
)
print(f"embedded {len(val) + len(all_train):,} photos in {time.perf_counter() - start:.0f} s")

# Each val photo's highest cosine to any train photo, 256 val photos at a time.
nearest = torch.cat(
    [(chunk @ train_embeddings.T).max(1).values for chunk in val_embeddings.split(256)]
)
nearest = nearest.clamp(-1, 1).cpu().numpy()  # rounding can take a copy just past 1
genuine, impostor = pair_histograms(val_embeddings, torch.as_tensor(val.labels, device=device))
del train_embeddings  # 0.9 GB of GPU memory
torch.cuda.empty_cache()

in_train = exact["have a copy in train"]
if in_train.any():
    print(f"lowest score of a val photo with an exact copy in train: {nearest[in_train].min():.4f}")
for threshold in (0.8, 0.9, 0.95, 0.99):
    count = (nearest >= threshold).sum()
    print(f"val photos with a cosine of at least {threshold} to some train photo: {count:,}")

fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
COARSE = 100  # EDGES' bins of 1e-4, summed 100 at a time into bins of 0.01
bins = np.r_[EDGES[::COARSE], 1.0]
for counts, label in (
    (genuine.reshape(-1, COARSE).sum(1), f"val genuine pairs ({genuine.sum():,.0f})"),
    (impostor.reshape(-1, COARSE).sum(1), f"val impostor pairs ({impostor.sum():,.0f})"),
    (
        np.histogram(nearest, bins)[0],
        f"each val photo's highest cosine to train ({len(nearest):,})",
    ),
):
    ax.stairs(counts / counts.sum(), bins, label=label)
ax.set(
    xlim=(-0.2, 1.01),
    yscale="log",
    xlabel="cosine",
    ylabel="share of its pairs or photos, per 0.01",
    title="L8: near copies",
)
ax.legend(loc="upper left")
save(fig, "L8")

## L9. Data loading (F4)

I try 0, 4 and 8 workers, with pinned memory on and off, and record GPU use and a torch.profiler
trace for each.

For each of the 6 settings I train softmax from a fresh model on batches of 512 from a shuffled
`DataLoader` over `train`, where each photo is one read from the decoded array on disk. The other
labs use `batches`, which takes photos from a copy in RAM and so skips the reading this lab
measures. With 0 workers the main process loads each batch between steps; with 4 or 8, worker
processes load the next batches while the GPU works on this one. Pinned memory makes the loader
put each batch in page-locked RAM, which the GPU can copy from directly, so the copy
(`non_blocking=True`) can run while the CPU carries on.

For each setting I measure: photos a second from the loader alone; photos a second training, over
25 steps after 5 to warm up, with the GPU's utilisation sampled by `nvidia-smi` every 100 ms; and a
profiler trace of 3 more steps, saved to `outputs/lab/` to open in ui.perfetto.dev.

In [ ]:
import subprocess

LOADER_BATCH = 512


def loader(workers, pinned):
    """A shuffled DataLoader over train, in batches of LOADER_BATCH, as training would use."""
    return DataLoader(
        train,
        batch_size=LOADER_BATCH,
        shuffle=True,
        num_workers=workers,
        pin_memory=pinned,
        drop_last=True,
        generator=torch.Generator().manual_seed(0),
    )


def loading_rate(workers, pinned, timed=40):
    """Photos a second the loader delivers with nothing else running.

    The first batch is left out: it includes starting the workers.
    """
    photos = iter(loader(workers, pinned))
    next(photos)
    start = time.perf_counter()
    for _ in range(timed):
        next(photos)
    return timed * LOADER_BATCH / (time.perf_counter() - start)


def gpu_sampler():
    """nvidia-smi printing the GPU's utilisation, in %, every 100 ms until stopped."""
    query = ["nvidia-smi", "--query-gpu=utilization.gpu", "--format=csv,noheader,nounits"]
    return subprocess.Popen([*query, "-lms", "100"], stdout=subprocess.PIPE, text=True)


def busy_share(sampler):
    """Stops a gpu_sampler; the mean of its readings, as a share."""
    sampler.terminate()
    return np.mean([int(reading) for reading in sampler.communicate()[0].split()]) / 100


def train_from_loader(workers, pinned, trace, warm=5, timed=25, profiled=3):
    """Softmax from a fresh model, fed by loader(workers, pinned).

    Runs warm steps untimed, then timed steps while sampling the GPU's utilisation, then profiled
    steps under the profiler, whose trace it saves to trace. Returns photos a second and the GPU's
    busy share over the timed steps.
    """
    model, head, optimiser = softmax_sgd(PICK)
    photos = iter(loader(workers, pinned))

    def step():
        images, labels = next(photos)
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        train_step(model, head, optimiser, images, labels)

    for _ in range(warm):
        step()
    torch.cuda.synchronize()
    sampler = gpu_sampler()
    start = time.perf_counter()
    for _ in range(timed):
        step()
    torch.cuda.synchronize()
    took = time.perf_counter() - start
    busy = busy_share(sampler)
    activities = [torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA]
    with torch.profiler.profile(activities=activities) as profiler:
        for _ in range(profiled):
            step()
        torch.cuda.synchronize()
    profiler.export_chrome_trace(str(trace))
    return timed * LOADER_BATCH / took, busy


l9 = {}
for workers in (0, 4, 8):
    for pinned in (False, True):
        name = f"{workers} workers, {'pinned' if pinned else 'not pinned'}"
        trace = RUNS / f"L9-{workers}-workers-{'pinned' if pinned else 'not-pinned'}.json"
        alone = loading_rate(workers, pinned)
        rate, busy = train_from_loader(workers, pinned, trace)
        l9[name] = alone, rate, busy
        print(
            f"{name}: loader alone {alone:,.0f} photos/s; training {rate:,.0f} photos/s, "
            f"GPU busy {busy:.0%}; trace in {trace.name}"
        )

In [ ]:
from matplotlib.ticker import PercentFormatter

fig, (rate_ax, busy_ax) = plt.subplots(1, 2, figsize=(13, 4.5), layout="constrained")
names = list(l9)
alone, rate, busy = np.array(list(l9.values())).T
rows = np.arange(len(names))
rate_ax.barh(rows - 0.2, alone, height=0.4, color="C0", label="loader alone")
rate_ax.barh(rows + 0.2, rate, height=0.4, color="C1", label="training")
rate_ax.set(yticks=rows, yticklabels=names, xscale="log", xlabel="photos a second")
rate_ax.invert_yaxis()  # first setting at the top
rate_ax.legend()
bars = busy_ax.barh(names, busy, color="C7")
busy_ax.bar_label(bars, labels=[f"{share:.0%}" for share in busy], padding=3)
busy_ax.invert_yaxis()
busy_ax.xaxis.set_major_formatter(PercentFormatter(1))
busy_ax.set(xlim=(0, 1.15), xlabel="time the GPU was running a kernel, while training")
fig.suptitle("L9: data loading")
save(fig, "L9")

## L10. Memory (F5)

I find the largest batch that fits, split peak memory into parameters, gradients, optimiser state
and activations, then try activation checkpointing.

A training step holds four things on the GPU: the parameters; their gradients; the optimiser's
state, which for SGD with momentum is one running sum per parameter; and the activations, each
layer's outputs kept from the forward pass because the backward pass needs them. The first three
don't depend on the batch; activations grow with every photo.

I measure one step's peak memory at 64, 128, 256 and 512 photos and split it: the first three are
their tensors' sizes, and activations are the rest of the peak. I do this with and without
activation checkpointing, which splits `model.body` into 4 segments of 2 blocks. For each segment
but the last, the forward pass keeps only the segment's input, and the backward pass runs the
segment forward again to get the rest. Then I find the largest batch that fits, each way, by
doubling the batch until a step runs out of memory, then halving the gap.

In [ ]:
import gc

from torch.utils.checkpoint import checkpoint_sequential


def memory_step(model, head, optimiser, images, labels, segments=None):
    """train_step, with model.body checkpointed in segments if given."""
    model.train()
    with torch.autocast(device.type, dtype=torch.bfloat16):
        x = model.stem(preprocess(images))
        if segments is None:
            x = model.body(x)
        else:
            x = checkpoint_sequential(model.body, segments, x, use_reentrant=False)
        embeddings = model.output(x)
    loss = cross_entropy_loss(head(embeddings.float(), labels), labels)
    optimiser.zero_grad()
    loss.backward()
    optimiser.step()
    return loss.item()


def size_gb(tensors):
    """The memory the tensors' values take, in GB."""
    return sum(t.numel() * t.element_size() for t in tensors) / 1e9


def memory_split(batch_size, segments=None):
    """The peak memory of a training step at batch_size, in GB, split four ways; and the step's
    time in seconds.

    The first step creates the gradients and the optimiser's momentum, so the second is measured.
    Activations are the rest of the peak, so they also hold the batch and temporary buffers.
    """
    held = torch.cuda.memory_allocated()  # what the notebook already holds
    model, head, optimiser = softmax_sgd(PICK)
    params = [*model.parameters(), *head.parameters()]
    images, labels = train_images[:batch_size].to(device), train_labels[:batch_size].to(device)
    memory_step(model, head, optimiser, images, labels, segments)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    memory_step(model, head, optimiser, images, labels, segments)
    torch.cuda.synchronize()
    took = time.perf_counter() - start
    split = {
        "parameters": size_gb(params),
        "gradients": size_gb(p.grad for p in params),
        "optimiser state": size_gb(
            t for state in optimiser.state.values() for t in state.values() if torch.is_tensor(t)
        ),
    }
    split["activations"] = (torch.cuda.max_memory_allocated() - held) / 1e9 - sum(split.values())
    return split, took


def fits(batch_size, segments=None):
    """Whether a training step at batch_size fits on the GPU."""
    try:
        memory_split(batch_size, segments)
        fitted = True
    except torch.OutOfMemoryError:
        fitted = False
    gc.collect()  # frees a failed step's tensors before the next try
    torch.cuda.empty_cache()
    return fitted


def largest_batch(segments=None, start=128, resolution=32, cap=2048):
    """The largest batch, to within resolution photos and up to cap, whose step fits on the GPU.

    Doubles the batch from start until a step runs out of memory, then halves the gap between the
    last that fitted and the first that didn't.
    """
    low, high = 0, start
    while high <= cap and fits(high, segments):
        low, high = high, 2 * high
    if high > cap:
        return low
    while high - low > resolution:
        middle = (low + high) // 2
        if fits(middle, segments):
            low = middle
        else:
            high = middle
    return low


gc.collect()
torch.cuda.empty_cache()
gpu_free, gpu_total = torch.cuda.mem_get_info()
print(
    f"GPU: {gpu_total / 1e9:.1f} GB, {gpu_free / 1e9:.1f} GB free; the notebook holds "
    f"{torch.cuda.memory_allocated() / 1e9:.2f} GB of tensors"
)
BATCHES = (64, 128, 256, 512)
SEGMENTS = {"plain": None, "checkpointed": 4}
l10, largest = {}, {}
for name, segments in SEGMENTS.items():
    l10[name] = {batch_size: memory_split(batch_size, segments) for batch_size in BATCHES}
    largest[name] = largest_batch(segments)
    split, took = l10[name][BATCHES[-1]]
    parts = ", ".join(f"{part} {gb:.2f}" for part, gb in split.items())
    print(f"{name}, {BATCHES[-1]} photos: {parts} GB; {1000 * took:.0f} ms a step")
    print(f"{name}: largest batch that fits {largest[name]}")

In [ ]:
fig, (split_ax, peak_ax) = plt.subplots(1, 2, figsize=(13, 4.5), layout="constrained")
names = list(l10)
left = np.zeros(len(names))
for colour, part in enumerate(l10[names[0]][BATCHES[-1]][0]):
    sizes = np.array([l10[name][BATCHES[-1]][0][part] for name in names])
    bars = split_ax.barh(names, sizes, left=left, color=f"C{colour}", label=part)
    left += sizes
split_ax.bar_label(
    bars,
    labels=[
        f"{gb:.1f} GB, {1000 * l10[name][BATCHES[-1]][1]:.0f} ms a step"
        for name, gb in zip(names, left, strict=True)
    ],
    padding=3,
)
split_ax.invert_yaxis()  # plain at the top
split_ax.margins(x=0.4)
split_ax.set(xlabel="GB at the step's peak", title=f"{BATCHES[-1]} photos a step")
split_ax.legend(loc="lower right")

for colour, name in enumerate(names):
    peaks = [sum(l10[name][batch_size][0].values()) for batch_size in BATCHES]
    label = f"{name}: largest batch {largest[name]}"
    peak_ax.plot(BATCHES, peaks, color=f"C{colour}", marker="o", label=label)
peak_ax.axhline(gpu_free / 1e9, color="grey", linestyle=":", label="free when the cell started")
peak_ax.set(xlabel="photos a step", ylabel="peak memory, GB", xticks=BATCHES, ylim=(0, None))
peak_ax.legend()
fig.suptitle("L10: memory")
save(fig, "L10")

## L11. Alignment (D1, F8)

I shift val faces by 2, 4 and 8 pixels and rotate them by 5 and 10 degrees, and plot TAR at FAR
1e-4 and 1e-3 against each.

I use the baseline (`outputs/lab/baseline.pt`). It trained on faces aligned to one template: eyes,
nose and mouth always on the same pixels, with about 35 pixels between the eyes. Each condition
moves every val photo on its own: a shift moves it 2, 4 or 8 pixels up, down, left or right, chosen
at random per photo; a rotation turns it 5 or 10 degrees, clockwise or anticlockwise at random. The
border it uncovers is black. The two photos of a pair move independently, so two photos shifted 8
pixels can end up 16 apart. "none" runs the same code without moving anything, so it should match
the baseline's eval-mode TAR in L5: 0.204 at FAR 1e-3.

In [ ]:
from torchvision.transforms.v2 import InterpolationMode
from torchvision.transforms.v2 import functional as TF

DIRECTIONS = torch.tensor([[1, 0], [-1, 0], [0, 1], [0, -1]])  # right, left, down, up


def misalign(shift=0, angle=0.0, seed=0):
    """A degrade for val_tar: each photo moved shift pixels in one of DIRECTIONS, and turned angle
    degrees clockwise or anticlockwise, each chosen at random per photo. The uncovered border is
    black.

    Whole-pixel shifts move pixels exactly; rotations interpolate bilinearly.
    """
    generator = torch.Generator().manual_seed(seed)

    def degrade(images):
        moves = DIRECTIONS[torch.randint(4, (len(images),), generator=generator)] * shift
        turns = (torch.randint(2, (len(images),), generator=generator) * 2 - 1) * angle
        moved = [
            TF.affine(
                image,
                angle=turn,
                translate=move,
                scale=1.0,
                shear=[0.0, 0.0],
                interpolation=InterpolationMode.BILINEAR,
            )
            for image, move, turn in zip(images, moves.tolist(), turns.tolist(), strict=True)
        ]
        return torch.stack(moved)

    return degrade


CONDITIONS = {  # name: shift in pixels, rotation in degrees
    "none": (0, 0.0),
    "shift 2 px": (2, 0.0),
    "shift 4 px": (4, 0.0),
    "shift 8 px": (8, 0.0),
    "rotate 5°": (0, 5.0),
    "rotate 10°": (0, 10.0),
}
model = baseline()
l11 = {}
for name, (shift, angle) in CONDITIONS.items():
    l11[name] = val_tar(model, degrade=misalign(shift, angle))
    print(f"{name}: TAR {l11[name][1e-4]:.3f} at FAR 1e-4, {l11[name][1e-3]:.3f} at FAR 1e-3")

In [ ]:
fig, (shift_ax, turn_ax) = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True, layout="constrained")
shifted = [(shift, name) for name, (shift, angle) in CONDITIONS.items() if angle == 0]
turned = [(angle, name) for name, (shift, angle) in CONDITIONS.items() if shift == 0]
for ax, points, colour in ((shift_ax, shifted, "C0"), (turn_ax, turned, "C1")):
    sizes = [size for size, _ in points]
    for far, style in ((1e-4, "-"), (1e-3, "--")):
        tars = [l11[name][far] for _, name in points]
        ax.plot(sizes, tars, color=colour, linestyle=style, marker="o", label=f"FAR {far:g}")
    ax.set(xticks=sizes)
    ax.legend()
shift_ax.set(xlabel="shift, pixels", ylabel="val TAR")
turn_ax.set(xlabel="rotation, degrees")
shift_ax.set_ylim(bottom=0)
fig.suptitle("L11: the baseline on misaligned val faces")
save(fig, "L11")